# 07 — ENSO & Ocean Scientific Analysis

Quantitative ocean diagnostics using GODAS and OISST data:

| Analysis | Method | Data |
|---|---|---|
| SST anomaly map | Monthly SST minus 1991–2020 climatology | OISST v2 |
| Niño index time series | Area-mean SST in monitoring boxes | OISST v2 |
| Thermocline depth (D20) | Depth of the 20 °C isotherm | GODAS pottmp |
| Warm Water Volume (WWV) | Volume integral of T > 20 °C (5°S–5°N) | GODAS pottmp |
| Equatorial depth–longitude section | Vertical T cross-section along the equator | GODAS pottmp |

---
**Stack:** `noawclg`, `xarray`, `numpy`, `scipy`, `matplotlib`, `cartopy`, `cmocean`

In [ ]:
import noawclg
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import copy
from scipy.interpolate import RegularGridInterpolator, interp1d

In [ ]:
# ── dark theme ────────────────────────────────────────────────────────────────
BG = "#0d1117"; LAND = "#13171d"; OCEAN_BG = "#0d1520"
COAST = "#2d6db5"; BORDER = "#2a2f36"
TEXT = "#e6edf3"; SUBTEXT = "#8b949e"
BOXBG = "#161b22"; BOXEDGE = "#30363d"
GRID = "#14181c"

plt.rcParams.update({
    "figure.facecolor":  BG,
    "axes.facecolor":    BG,
    "savefig.facecolor": BG,
    "text.color":        TEXT,
    "axes.labelcolor":   SUBTEXT,
    "xtick.color":       SUBTEXT,
    "ytick.color":       SUBTEXT,
    "axes.edgecolor":    BOXEDGE,
    "axes.grid":         True,
    "grid.color":        GRID,
    "grid.linewidth":    0.5,
    "font.family":       "monospace",
    "legend.facecolor":  BOXBG,
    "legend.edgecolor":  BOXEDGE,
    "legend.labelcolor": TEXT,
})

In [ ]:
# ── helper: upsample a coarse grid ───────────────────────────────────────────
def upsample(lat, lon, field, factor=4):
    interp = RegularGridInterpolator(
        (lat, lon), field, method="linear",
        bounds_error=False, fill_value=np.nan,
    )
    lat_f = np.linspace(lat[0], lat[-1], len(lat) * factor)
    lon_f = np.linspace(lon[0], lon[-1], len(lon) * factor)
    lg, la = np.meshgrid(lon_f, lat_f)
    return lat_f, lon_f, interp((la, lg))


# ── helper: GODAS → (lat, lon_180, field) ────────────────────────────────────
def load_godas(var, year, month, depth_m=None):
    lat, lon, field = noawclg.get_godas(var, year, month, depth_m=depth_m)
    lon_180 = np.where(lon > 180, lon - 360, lon)
    order   = np.argsort(lon_180)
    return lat, lon_180[order], field[:, order]


# ── helper: OISST → (lat, lon_180, sst) ──────────────────────────────────────
def load_oisst(year, month):
    lat, lon, sst = noawclg.open_oisst(year, month)
    return lat, lon, sst

---
## 1  SST Anomaly Map

Anomaly = current month SST minus the 1991–2020 climatological mean for that month.

> **Tip:** building a 30-year climatology requires loading 30 × 12 months of data.  
> Here we approximate using the last 5 years for speed — replace with 30 years for research-grade results.

In [ ]:
TARGET_YEAR  = 2024
TARGET_MONTH = 5
CLIM_YEARS   = range(2019, 2024)   # 5-year approximation (use 1991–2020 for proper climatology)

# Target SST
lat_s, lon_s, sst_now = load_oisst(TARGET_YEAR, TARGET_MONTH)

# Climatological mean for the same month
clim_stack = []
for yr in CLIM_YEARS:
    _, _, sst_yr = load_oisst(yr, TARGET_MONTH)
    clim_stack.append(sst_yr)

sst_clim = np.nanmean(clim_stack, axis=0)
sst_anom = sst_now - sst_clim

print(f"Anomaly range: {np.nanmin(sst_anom):.2f} … {np.nanmax(sst_anom):.2f} °C")

In [ ]:
# ── Plot on a PlateCarrée tropical belt ───────────────────────────────────────
levels_a = np.arange(-3.0, 3.25, 0.25)
cmap_a   = copy.copy(cmocean.cm.balance)
cmap_a.set_bad(alpha=0); cmap_a.set_under(alpha=0)
norm_a   = mcolors.BoundaryNorm(levels_a, ncolors=cmap_a.N, clip=False)

proj = ccrs.PlateCarree()
fig, ax = plt.subplots(figsize=(16, 6),
                        subplot_kw={"projection": proj}, facecolor=BG)
ax.set_facecolor(BG)
ax.set_extent([-180, 180, -60, 60], crs=ccrs.PlateCarree())

ax.add_feature(cfeature.LAND.with_scale("110m"),     facecolor=LAND,  zorder=2)
ax.add_feature(cfeature.COASTLINE.with_scale("110m"),
               edgecolor=COAST, linewidth=0.5, zorder=3)

cf = ax.pcolormesh(lon_s, lat_s, sst_anom,
                   cmap=cmap_a, norm=norm_a,
                   transform=ccrs.PlateCarree(), zorder=1)

# Contour the ±0.5 °C isoline
ax.contour(lon_s, lat_s, sst_anom,
           levels=[-0.5, 0.5], colors=["#58a6ff", "#f85149"],
           linewidths=0.8, linestyles="--",
           transform=ccrs.PlateCarree(), zorder=4)

# ENSO monitoring boxes
NINO_BOXES = {
    "3.4": dict(lon=(-170, -120), lat=(-5, 5), color="#f7c948"),
    "4":   dict(lon=(-200+360, -150+360), lat=(-5, 5), color="#58a6ff"),
}
for name, box in NINO_BOXES.items():
    w = box["lon"][0] if box["lon"][0] <= 180 else box["lon"][0] - 360
    e = box["lon"][1] if box["lon"][1] <= 180 else box["lon"][1] - 360
    s, n = box["lat"]
    ax.plot([w, e, e, w, w], [s, s, n, n, s],
            color=box["color"], linewidth=1.2, linestyle="-",
            transform=ccrs.PlateCarree(), zorder=5)
    ax.text((w + e) / 2, n + 1.5, f"Niño {name}",
            color=box["color"], fontsize=7, ha="center",
            fontweight="bold",
            transform=ccrs.PlateCarree(), zorder=6)

gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                  linewidth=0.3, color=GRID, linestyle="--")
gl.top_labels = False; gl.right_labels = False
gl.xlabel_style = {"color": SUBTEXT, "fontsize": 7}
gl.ylabel_style = {"color": SUBTEXT, "fontsize": 7}

cb = fig.colorbar(cf, ax=ax, orientation="vertical",
                  pad=0.02, fraction=0.015, shrink=0.9)
cb.set_label("SST Anomaly (°C)", fontsize=8, color=SUBTEXT)
cb.ax.tick_params(labelsize=7, colors=SUBTEXT)
cb.outline.set_edgecolor(BOXEDGE)

ax.set_title(f"SST Anomaly vs {min(CLIM_YEARS)}–{max(CLIM_YEARS)} mean",
             fontsize=10, fontweight="bold", color=TEXT, loc="left", pad=6)
ax.set_title(f"OISST v2 · {TARGET_YEAR}-{TARGET_MONTH:02d}",
             fontsize=7, color=SUBTEXT, loc="right", pad=6)

fig.tight_layout()
fig.savefig("sst_anomaly.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 2  Niño Index Time Series

The **Niño 3.4 index** is the area-mean SST anomaly in the box 5°S–5°N, 170°W–120°W.  
Values > +0.5 °C for ≥ 5 consecutive months = El Niño; < −0.5 °C = La Niña.

In [ ]:
def area_mean_sst_anom(year, month, lat_arr, lon_arr, clim_map,
                        lat_box, lon_box):
    """Load OISST for (year, month) and return area-mean anomaly inside the box."""
    _, _, sst = load_oisst(year, month)
    anom = sst - clim_map

    lat_mask = (lat_arr >= lat_box[0]) & (lat_arr <= lat_box[1])
    lon_mask = (lon_arr >= lon_box[0]) & (lon_arr <= lon_box[1])
    subset   = anom[np.ix_(lat_mask, lon_mask)]
    return float(np.nanmean(subset))


# Build a 5-year climatology for the whole globe once per month
# (in a real study you'd use 1991–2020 from ERSST or OISST)
ANALYSIS_YEARS  = range(2020, 2025)    # 5 years of monthly data
MONTHS          = range(1, 13)

# Pre-load climatology per calendar month
clim_by_month = {}
for m in MONTHS:
    stack = []
    for yr in CLIM_YEARS:
        _, _, sst_yr = load_oisst(yr, m)
        stack.append(sst_yr)
    clim_by_month[m] = np.nanmean(stack, axis=0)

# Niño 3.4 box: 5°S–5°N, 170°W–120°W
NINO34_LAT = (-5,  5)
NINO34_LON = (-170, -120)

nino34_index = []
time_labels  = []

for yr in ANALYSIS_YEARS:
    for m in MONTHS:
        val = area_mean_sst_anom(
            yr, m, lat_s, lon_s, clim_by_month[m],
            NINO34_LAT, NINO34_LON,
        )
        nino34_index.append(val)
        time_labels.append(f"{yr}-{m:02d}")

nino34_index = np.array(nino34_index)
print(f"Max Niño 3.4: {nino34_index.max():.2f} °C  ({time_labels[nino34_index.argmax()]})")
print(f"Min Niño 3.4: {nino34_index.min():.2f} °C  ({time_labels[nino34_index.argmin()]})")

In [ ]:
fig2, ax2 = plt.subplots(figsize=(14, 4), facecolor=BG)
ax2.set_facecolor(BG)

x = np.arange(len(nino34_index))
xtick_step = 3   # label every 3 months

# Filled areas: El Niño (red) / La Niña (blue)
ax2.fill_between(x, nino34_index, 0,
                 where=nino34_index > 0,
                 color="#f85149", alpha=0.35, label="El Niño (warm)")
ax2.fill_between(x, nino34_index, 0,
                 where=nino34_index < 0,
                 color="#58a6ff", alpha=0.35, label="La Niña (cool)")

ax2.plot(x, nino34_index, color="#e6edf3", linewidth=1.2, zorder=3)
ax2.axhline(+0.5, color="#f85149", linewidth=0.8, linestyle="--", alpha=0.7)
ax2.axhline(-0.5, color="#58a6ff", linewidth=0.8, linestyle="--", alpha=0.7)
ax2.axhline(0,    color=BOXEDGE,  linewidth=0.6)

ax2.set_xticks(x[::xtick_step])
ax2.set_xticklabels(time_labels[::xtick_step], rotation=45, ha="right", fontsize=7)
ax2.set_ylabel("SST Anomaly (°C)", fontsize=8)
ax2.set_title("Niño 3.4 Index (5°S–5°N, 170°W–120°W)",
              fontsize=10, fontweight="bold", color=TEXT, loc="left")
ax2.set_title("OISST v2  |  threshold ±0.5 °C",
              fontsize=7, color=SUBTEXT, loc="right")
ax2.legend(fontsize=8, loc="upper left")
ax2.set_ylim(-3, 3)

fig2.tight_layout()
fig2.savefig("nino34_index.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 3  Thermocline Depth — D20

The **depth of the 20 °C isotherm (D20)** tracks the base of the warm upper ocean layer.  
El Niño deepens D20 in the eastern Pacific; La Niña shallows it.

Method: for each (lat, lon) grid point, vertically interpolate GODAS `pottmp` and find the depth at which T = 20 °C.

In [ ]:
# GODAS depth levels (m) — all 40 levels
GODAS_LEVELS = np.array([
     5, 15, 25, 35, 45, 55, 65, 75, 85, 95,
    105,115,125,135,145,155,165,175,185,195,
    205,215,225,238,262,303,366,459,584,747,
    949,1193,1479,1807,2174,2579,3016,3483,3972,4478,
], dtype=float)


def compute_d20(pottmp_3d, depths, target_temp=20.0):
    """
    pottmp_3d : (n_levels, n_lat, n_lon) in °C
    depths    : 1D array of depth levels (m), monotonically increasing
    Returns   : 2D array (n_lat, n_lon) — depth of target_temp isotherm in m
    """
    n_lev, n_lat, n_lon = pottmp_3d.shape
    d20 = np.full((n_lat, n_lon), np.nan)

    for i in range(n_lat):
        for j in range(n_lon):
            profile = pottmp_3d[:, i, j]
            valid   = ~np.isnan(profile)
            if valid.sum() < 3:
                continue
            t_v = profile[valid]
            z_v = depths[valid]
            # Temperature must decrease with depth near the thermocline
            # Find first crossing below 20 °C
            crossings = np.where(
                (t_v[:-1] >= target_temp) & (t_v[1:] < target_temp)
            )[0]
            if len(crossings) == 0:
                continue
            k    = crossings[0]
            frac = (t_v[k] - target_temp) / (t_v[k] - t_v[k + 1])
            d20[i, j] = z_v[k] + frac * (z_v[k + 1] - z_v[k])

    return d20

In [ ]:
# Load all depth levels for pottmp
pottmp_levels = []
for depth_m in GODAS_LEVELS:
    lat_g, lon_g, pt = load_godas("pottmp", TARGET_YEAR, TARGET_MONTH, depth_m=depth_m)
    pottmp_levels.append(pt - 273.15)   # K → °C

pottmp_3d = np.stack(pottmp_levels, axis=0)  # (40, n_lat, n_lon)
print("3D pottmp shape:", pottmp_3d.shape)

In [ ]:
d20 = compute_d20(pottmp_3d, GODAS_LEVELS)
print(f"D20 range: {np.nanmin(d20):.0f} … {np.nanmax(d20):.0f} m")

# Upsample for smoother rendering
lat_d, lon_d, d20_fine = upsample(lat_g, lon_g, d20, factor=3)

In [ ]:
levels_d20 = np.array([20, 40, 60, 80, 100, 120, 140, 160, 180, 200, 220, 250, 300])
cmap_d20   = copy.copy(cmocean.cm.deep)
cmap_d20.set_bad(alpha=0); cmap_d20.set_under(alpha=0)
norm_d20   = mcolors.BoundaryNorm(levels_d20, ncolors=cmap_d20.N, clip=False)

proj_d = ccrs.PlateCarree()
fig3, ax3 = plt.subplots(figsize=(16, 5),
                          subplot_kw={"projection": proj_d}, facecolor=BG)
ax3.set_facecolor(BG)
ax3.set_extent([-180, 180, -30, 30], crs=ccrs.PlateCarree())

ax3.add_feature(cfeature.LAND.with_scale("110m"),     facecolor=LAND,  zorder=2)
ax3.add_feature(cfeature.COASTLINE.with_scale("110m"),
                edgecolor=COAST, linewidth=0.5, zorder=3)

cf3 = ax3.pcolormesh(lon_d, lat_d, d20_fine,
                     cmap=cmap_d20, norm=norm_d20,
                     transform=ccrs.PlateCarree(), zorder=1)

# Contour the 100 m and 150 m isolines
ax3.contour(lon_d, lat_d, d20_fine,
            levels=[100, 150], colors=["#e6edf3"],
            linewidths=0.7, alpha=0.6,
            transform=ccrs.PlateCarree(), zorder=4)

gl3 = ax3.gridlines(draw_labels=True, linewidth=0.3, color=GRID, linestyle="--")
gl3.top_labels = False; gl3.right_labels = False
gl3.xlabel_style = {"color": SUBTEXT, "fontsize": 7}
gl3.ylabel_style = {"color": SUBTEXT, "fontsize": 7}

cb3 = fig3.colorbar(cf3, ax=ax3, orientation="vertical",
                    pad=0.02, fraction=0.015, shrink=0.9)
cb3.set_label("D20 depth (m)", fontsize=8, color=SUBTEXT)
cb3.ax.tick_params(labelsize=7, colors=SUBTEXT)
cb3.outline.set_edgecolor(BOXEDGE)

ax3.set_title("Thermocline Depth — D20 (depth of 20 °C isotherm)",
              fontsize=10, fontweight="bold", color=TEXT, loc="left", pad=6)
ax3.set_title(f"GODAS · {TARGET_YEAR}-{TARGET_MONTH:02d}",
              fontsize=7, color=SUBTEXT, loc="right", pad=6)

fig3.tight_layout()
fig3.savefig("d20_thermocline.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4  Warm Water Volume (WWV)

WWV = volume of ocean water warmer than 20 °C between 5°S–5°N, 120°E–80°W.  
It is a leading indicator of El Niño — WWV increases 6–12 months before warm events.

In [ ]:
# WWV box (in 0–360 convention for GODAS lon before conversion)
WWV_LAT = (-5, 5)
WWV_LON = (-180, -80)   # 120°E = −180°+300° ... use −180 to −80 after lon conversion

# Approximate cell areas at equatorial latitudes (degrees²)
def wwv_from_d20(d20_2d, lat_1d, lon_1d,
                  lat_box=WWV_LAT, lon_box=WWV_LON):
    """Estimate WWV as the mean D20 × area of the box (proportional to volume)."""
    lat_mask = (lat_1d >= lat_box[0]) & (lat_1d <= lat_box[1])
    lon_mask = (lon_1d >= lon_box[0]) & (lon_1d <= lon_box[1])
    subset   = d20_2d[np.ix_(lat_mask, lon_mask)]
    return float(np.nanmean(subset))   # mean D20 depth in the box (m)


# Monthly WWV proxy (mean D20 in the box) over the analysis period
wwv_series = []

for yr in ANALYSIS_YEARS:
    for m in MONTHS:
        # Load all levels
        pt_levels = []
        for depth_m in GODAS_LEVELS[:20]:   # shallow 20 levels are enough for D20
            _lat, _lon, _pt = load_godas("pottmp", yr, m, depth_m=depth_m)
            pt_levels.append(_pt - 273.15)
        pt3d = np.stack(pt_levels, axis=0)
        d20_m = compute_d20(pt3d, GODAS_LEVELS[:20])
        wwv_series.append(wwv_from_d20(d20_m, _lat, _lon))

wwv_series = np.array(wwv_series)
wwv_anom   = wwv_series - wwv_series.mean()

print(f"Mean D20 in WWV box: {wwv_series.mean():.1f} m")

In [ ]:
fig4, axes4 = plt.subplots(2, 1, figsize=(14, 6),
                             sharex=True, facecolor=BG)

x = np.arange(len(wwv_series))

# Top: absolute mean D20
axes4[0].plot(x, wwv_series, color="#58a6ff", linewidth=1.4)
axes4[0].fill_between(x, wwv_series, wwv_series.mean(),
                       where=wwv_series > wwv_series.mean(),
                       color="#f85149", alpha=0.3, label="Above mean (warm)")
axes4[0].fill_between(x, wwv_series, wwv_series.mean(),
                       where=wwv_series < wwv_series.mean(),
                       color="#58a6ff", alpha=0.3, label="Below mean (cool)")
axes4[0].axhline(wwv_series.mean(), color=BOXEDGE, linewidth=0.8, linestyle="--")
axes4[0].set_ylabel("Mean D20 (m)", fontsize=8)
axes4[0].set_title("Warm Water Volume proxy — mean D20 in WWV box (5°S–5°N, 120°E–80°W)",
                   fontsize=9, fontweight="bold", color=TEXT, loc="left")
axes4[0].legend(fontsize=7, loc="upper right")
axes4[0].invert_yaxis()   # deeper = warmer

# Bottom: anomaly
axes4[1].fill_between(x, wwv_anom, 0,
                       where=wwv_anom > 0,
                       color="#f85149", alpha=0.4)
axes4[1].fill_between(x, wwv_anom, 0,
                       where=wwv_anom < 0,
                       color="#58a6ff", alpha=0.4)
axes4[1].plot(x, wwv_anom, color="#e6edf3", linewidth=1.2)
axes4[1].axhline(0, color=BOXEDGE, linewidth=0.8)
axes4[1].set_ylabel("Anomaly (m)", fontsize=8)
axes4[1].set_xticks(x[::3])
axes4[1].set_xticklabels(time_labels[::3], rotation=45, ha="right", fontsize=7)
axes4[1].set_title("D20 anomaly (positive = deeper thermocline = more warm water)",
                   fontsize=8, color=SUBTEXT, loc="left")

for ax_ in axes4:
    ax_.set_facecolor(BG)

fig4.tight_layout()
fig4.savefig("wwv_d20.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 5  Equatorial Depth–Longitude Cross-Section

A vertical slice of temperature along the equator (averaged 2°S–2°N) shows how heat content distributes horizontally with depth — the signature of ENSO is visible as a tilt in the thermocline.

In [ ]:
# Use only the upper 500 m (levels with index < 30)
SHALLOW_IDX    = GODAS_LEVELS < 500
shallow_depths = GODAS_LEVELS[SHALLOW_IDX]

EQ_LAT_BAND = (-2, 2)   # average over 2°S–2°N

cross_section = []

for depth_m in shallow_depths:
    lat_g, lon_g, pt = load_godas("pottmp", TARGET_YEAR, TARGET_MONTH, depth_m=depth_m)
    pt_c = pt - 273.15

    lat_mask = (lat_g >= EQ_LAT_BAND[0]) & (lat_g <= EQ_LAT_BAND[1])
    eq_slice = np.nanmean(pt_c[lat_mask, :], axis=0)   # (n_lon,)
    cross_section.append(eq_slice)

cross_section = np.array(cross_section)   # (n_shallow_levels, n_lon)
print("Cross-section shape:", cross_section.shape)

In [ ]:
levels_cs = np.arange(2, 32, 2)
cmap_cs   = copy.copy(cmocean.cm.thermal)
cmap_cs.set_bad(alpha=0); cmap_cs.set_under(alpha=0)
norm_cs   = mcolors.BoundaryNorm(levels_cs, ncolors=cmap_cs.N, clip=False)

fig5, ax5 = plt.subplots(figsize=(14, 5), facecolor=BG)
ax5.set_facecolor(BG)

# pcolormesh: x = longitude, y = depth (inverted)
cf5 = ax5.pcolormesh(
    lon_g, shallow_depths, cross_section,
    cmap=cmap_cs, norm=norm_cs,
    shading="auto",
)

# 20 °C isotherm — the thermocline
ax5.contour(
    lon_g, shallow_depths, cross_section,
    levels=[20], colors=["#f7c948"],
    linewidths=1.5, linestyles="-",
)
ax5.contour(
    lon_g, shallow_depths, cross_section,
    levels=levels_cs[::2], colors="white",
    linewidths=0.3, alpha=0.35,
)

ax5.invert_yaxis()
ax5.set_xlabel("Longitude (°)", fontsize=8)
ax5.set_ylabel("Depth (m)", fontsize=8)

# Mark Pacific basin boundaries
ax5.axvline(-180, color=BORDER, linewidth=0.8, linestyle=":")
ax5.axvline(-80,  color=BORDER, linewidth=0.8, linestyle=":")
ax5.text(-130, 30, "Tropical Pacific",
         color=SUBTEXT, fontsize=7, ha="center", va="top")

cb5 = fig5.colorbar(cf5, ax=ax5, orientation="vertical",
                    pad=0.01, fraction=0.015, shrink=0.95)
cb5.set_label("Temperature (°C)", fontsize=8, color=SUBTEXT)
cb5.ax.tick_params(labelsize=7, colors=SUBTEXT)
cb5.outline.set_edgecolor(BOXEDGE)

ax5.set_title("Equatorial Temperature Cross-Section (2°S–2°N avg) — yellow = 20 °C isotherm",
              fontsize=9, fontweight="bold", color=TEXT, loc="left")
ax5.set_title(f"GODAS · {TARGET_YEAR}-{TARGET_MONTH:02d}",
              fontsize=7, color=SUBTEXT, loc="right")

fig5.tight_layout()
fig5.savefig("equatorial_cross_section.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Exercises

1. **Extend the climatology** — replace `CLIM_YEARS = range(2019, 2024)` with `range(1991, 2021)` and re-run the SST anomaly. Requires 30 OISST downloads but gives a proper WMO climatological mean.
2. **All four Niño boxes** — compute Niño 1+2, 3, 3.4, and 4 on the same subplot and compare their timing.
3. **Hovmöller diagram** — loop over months and stack the equatorial mean SST as rows, then plot a (time × longitude) pcolormesh. This reveals the eastward propagation of warm anomalies.
4. **Salinity cross-section** — load GODAS `salt` the same way and plot the halocline alongside the thermocline.
5. **D20 anomaly map** — compute D20 for the same month one year earlier and subtract to isolate the interannual signal.